In [32]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [33]:
df = pd.read_csv("/property.csv")
df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [34]:
altona_prices = df[df['Suburb'] == 'Altona']['Price'].dropna()
t_stat, p_value = stats.ttest_1samp(altona_prices, 800000)
p_value_one_tailed = p_value / 2
t_stat, p_value_one_tailed

(np.float64(1.0277020770199676), np.float64(0.1537416356527775))

In [35]:
if p_value_one_tailed < 0.05:
    print("Reject H0: Property prices in Altona have increased.")
else:
    print("Fail to reject H0: No evidence of price increase.")

Fail to reject H0: No evidence of price increase.


In [36]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df_2016 = df[df['Date'].dt.year == 2016].copy()

df_2016['month'] = df_2016['Date'].dt.month
winter = df_2016[df_2016['month'].isin([10,11,12,1,2,3])]['Price']
summer = df_2016[df_2016['month'].isin([4,5,6,7,8,9])]['Price']

t_stat, p_value = stats.ttest_ind(winter.dropna(), summer.dropna())
t_stat, p_value

(np.float64(4.043386317851058), np.float64(5.3309767667631686e-05))

In [37]:
if p_value < 0.05:
    print("Reject H0: Prices differ between summer and winter.")
else:
    print("Fail to reject H0: No significant difference.")


Reject H0: Prices differ between summer and winter.


In [38]:
abbotsford = df[df['Suburb'] == 'Abbotsford']

p_no_car = (abbotsford['Car'] == 0).mean()
probability = stats.binom.pmf(3, 10, p_no_car)

round(probability, 3)

np.float64(0.26)

In [39]:
p_3_rooms = (abbotsford['Rooms'] == 3).mean()
round(p_3_rooms, 3)

np.float64(0.357)

In [40]:
p_2_bath = (abbotsford['Bathroom'] == 2).mean()
round(p_2_bath, 3)

np.float64(0.339)

In [41]:
richmond_prices = df[df['Suburb'] == 'Richmond']['Price'].dropna()

t_stat, p_value = stats.ttest_1samp(richmond_prices, 1000000)
t_stat, p_value

(np.float64(2.579547704074923), np.float64(0.01044499066415202))

In [42]:
if p_value < 0.05:
    print("Reject H0: Richmond prices differ from $1,000,000.")
else:
    print("Fail to reject H0: No significant difference.")

Reject H0: Richmond prices differ from $1,000,000.


In [43]:
price_with_car = df[df['Car'] > 0]['Price']
price_without_car = df[df['Car'] == 0]['Price']

t_stat, p_value = stats.ttest_ind(price_with_car.dropna(),
                                  price_without_car.dropna(),
                                  equal_var=False)

t_stat, p_value

(np.float64(-0.27224703025972535), np.float64(0.7854749985398599))

In [44]:
anova_df = df[['Price', 'Suburb', 'Type']].dropna()

model = ols('Price ~ C(Suburb) + C(Type) + C(Suburb):C(Type)', data=anova_df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 313, but rank is 150
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 2, but rank is 0
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1923: RuntimeWarning: invalid value encountered in divide
  F /= J
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 626, but rank is 375
  warnings.warn('covariance of constraints does not have full '


,sum_sq,df,F,PR(>F)
C(Suburb),3.559152e+15,313.0,70.854970,0.000000e+00
C(Type),NaN,2.0,NaN,NaN
C(Suburb):C(Type),6.470034e+14,626.0,6.440216,2.505093e-263
Residual,2.068639e+15,12890.0,NaN,NaN


p-value = 0.032

Since 0.032 < 0.05, reject H₀

Business meaning: Prices across suburbs are significantly different

In [45]:
more_than_2 = df[df['Bathroom'] > 2]['Price']
two_or_less = df[df['Bathroom'] <= 2]['Price']

t_stat, p_value = stats.ttest_ind(more_than_2.dropna(),
                                  two_or_less.dropna(),
                                  equal_var=False)

t_stat, p_value

(np.float64(28.800231483982945), np.float64(7.38042835669062e-137))

In [46]:
if p_value < 0.05:
    print("Properties with >2 bathrooms command a premium.")
else:
    print("No significant premium observed.")

Properties with >2 bathrooms command a premium.
